In [8]:
# PDF -> per-page text extraction (writes to resources/stormlight_texts)
from pathlib import Path
from pypdf import PdfReader

def convert_pdf_to_text_files(pdf_path: Path | str, output_dir: str = "resources/stormlight_texts") -> list[dict]:
    pdf_path = Path(pdf_path)
    out_path = Path(output_dir)
    out_path.mkdir(parents=True, exist_ok=True)
    reader = PdfReader(str(pdf_path))
    texts = []
    pages = getattr(reader, 'pages', None) or []
    for i, page in enumerate(pages, start=1):
        try:
            # pypdf/PyPDF2 expose extract_text on page objects
            text = page.extract_text() or ""
        except Exception:
            text = ""
        filename = f"page_{i:04d}.txt"
        (out_path / filename).write_text(text, encoding="utf-8")
        texts.append({"title": filename, "path": str(out_path / filename), "text": text})
    print(f"Extracted {len(texts)} pages to {out_path.resolve()}")
    return texts

# Locate PDF in resources/ and extract
pdf_candidates = list(Path("resources").glob("*.pdf"))
if not pdf_candidates:
    raise FileNotFoundError("No PDF found in resources/. Put your Stormlight PDF in resources/ and re-run this cell.")
pdf_path = pdf_candidates[0]
print(f"Using PDF: {pdf_path}")
stormlight_documents = convert_pdf_to_text_files(pdf_path, output_dir="resources/stormlight_texts")
# `stormlight_documents` is a list of {title,path,text} dicts for downstream processing

Using PDF: resources\SL001_Stormlight_Handbook_digital.pdf
Extracted 392 pages to C:\Users\scott\projects\llm_engineering\my-stuff\stormlight-role-playing-game\resources\stormlight_texts


In [6]:
 # Chunk texts, create embeddings, and persist a Chroma vector store
from pathlib import Path
from dotenv import load_dotenv

# Import embedding and vector store libraries (adjust imports to your environment)
try:
    from langchain_openai import OpenAIEmbeddings
    from langchain_chroma import Chroma
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    from langchain_core.documents import Document
except Exception:
    # Fallback: try common package names; if these fail, install appropriate packages
    raise ImportError("Required langchain_openai/langchain_chroma/langchain_text_splitters not found in environment.")

load_dotenv(override=True)

# Locate extracted texts directory
candidate_paths = [
    Path('resources/stormlight_texts'),
    Path.cwd() / 'resources' / 'stormlight_texts',
]
tale_dir = next((p for p in candidate_paths if p.exists()), None)
assert tale_dir is not None, f"Extracted text directory not found. Checked: {candidate_paths}"
repo_root = tale_dir.parent.parent
print(f"Using text directory: {tale_dir.resolve()}")

# Collect text files into documents (if stormlight_documents exists, prefer it)
docs = []
if 'stormlight_documents' in globals():
    source_docs = stormlight_documents
else:
    file_paths = sorted(tale_dir.glob('*.txt'))
    source_docs = []
    for p in file_paths:
        source_docs.append({'title': p.stem, 'path': str(p), 'text': p.read_text(encoding='utf-8')})

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunk_docs = []
for doc in source_docs:
    chunks = splitter.split_text(doc['text'])
    for i, chunk in enumerate(chunks):
        chunk_docs.append(Document(page_content=chunk, metadata={'title': doc.get('title'), 'source': doc.get('path'), 'chunk': i}))

print(f'Created {len(chunk_docs)} chunks for embedding')


embedding_model = OpenAIEmbeddings(model='text-embedding-3-large')
vector_db_dir = repo_root / 'resources' / 'stormlight_vector_db'
vector_db = Chroma.from_documents(
    documents=chunk_docs,
    embedding=embedding_model,
    persist_directory=str(vector_db_dir),
    collection_name='stormlight',
)
print(f'Built or loaded Chroma vector store at: {vector_db_dir.resolve()}')

Using text directory: C:\Users\scott\projects\llm_engineering\my-stuff\stormlight-role-playing-game\resources\stormlight_texts
Created 1643 chunks for embedding
Built or loaded Chroma vector store at: C:\Users\scott\projects\llm_engineering\my-stuff\stormlight-role-playing-game\resources\stormlight_vector_db


In [7]:
# Build retriever and RAG answer chain using the persisted vector store
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.runnables import RunnableLambda

# Load or reuse vector_db from previous cell
retriever = vector_db.as_retriever(search_kwargs={"k": 4})

llm = ChatOpenAI(model_name="gpt-4.1-nano", temperature=0)

def make_answer_chain(question: str):
    def docs_to_messages(docs):
        if not docs:
            return [
                SystemMessage(
                    "You are an expert assistant answering questions about the Stormlight Handbook. Use only the retrieved context to answer the user clearly and accurately. If the answer is not present in the context, say you don't know."
                ),
                HumanMessage(content=f"Question: {question}\n\nContext:\nNo relevant context was found."),
            ]

        context = "\n\n".join(f"Source: {doc.metadata.get('title', doc.metadata.get('source', 'unknown'))}\n{doc.page_content}" for doc in docs)
        
        return [
            SystemMessage(
                "You are an expert assistant answering questions about the Stormlight Handbook. Use only the retrieved context to answer the user clearly and accurately. If the answer is not present in the context, say you don't know."
            ),
            HumanMessage(content=f"Question: {question}\n\nContext:\n{context}"),
        ]

    return retriever | RunnableLambda(docs_to_messages) | llm

def answer_stormlight_question(question: str) -> str:
    chain = make_answer_chain(question)
    response = chain.invoke(question)
    return response.content

# Example query (replace with your own)
print(answer_stormlight_question("What is a shardblade?"))

A Shardblade is a weapon composed of an unknown metal with a gemstone fitted within its hilt or guard. It can cut cleanly through almost any non-living material, including stone and metal, with ease. Additionally, it passes through living organisms without injury, slicing through their soul rather than flesh and bone. When a Shardblade severs a limb, everything beyond the cut becomes Blade-dead, lifeless and gray. If it cuts the spine or head, the person instantly dies.


In [15]:
# Sample queries to verify retrieval and answers
questions = [
    "What is a shardblade?",
    "Describe the different orders of the Knights Radiant.",
    "What are spren?",
    "How does one acquire Stormlight?",
    "What is the purpose of a Shardplate?",
]

for q in questions:
    answer = answer_stormlight_question(q)
    print('Answer:\n', answer)

Answer:
 A Shardblade is a weapon composed of an unknown metal with a gemstone fitted within its hilt or guard. It can cut through almost any non-living material with ease and pass through living organisms without injury, slicing through their soul rather than flesh and bone. When a Shardblade severs a limb, everything beyond the cut becomes Blade-dead, lifeless and gray. If it cuts the spine or head, the person instantly dies. Historically, Shardblades are known to be the corpses of Radiant spren, killed when their bonded knights forswore their oaths during the Recreance.
Answer:
 The different orders of the Knights Radiant are as follows:

1. **Dustbringers**: Bond with Ashspren, command the Surges of Abrasion and Division, and believe that great power requires strong discipline.
2. **Edgedancers**: Bond with Cultivationspren, command the Surges of Abrasion and Progression, and focus on remembering and serving those who others forget.
3. **Elsecallers**: Bond with Inkspren, command t

In [8]:
import gradio as gr

def gradio_answer(question: str) -> str:
    return answer_stormlight_question(question)

qa_interface = gr.Interface(
    fn=gradio_answer,
    inputs=gr.Textbox(lines=2, placeholder="Ask a question about the Stormlight Handbook...", label="Question"),
    outputs=gr.Textbox(label="Answer", lines=12),
    title="Stormlight Handbook QA",
    description="Retriever-augmented question answering over the Stormlight Handbook texts.",
    allow_flagging="never",
)

# Launch the interface (set share=True if you want an external link)
qa_interface.launch(share=False)

c:\Users\scott\projects\llm_engineering\.venv\Lib\site-packages\gradio\interface.py:415: UserWarning: The `allow_flagging` parameter in `Interface` is deprecated. Use `flagging_mode` instead.
  warnings.warn(


* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
